# Symbolic Regression with PySR


## Background

### Symbolic Regression
Symbolic regression is a machine learning task where the goal is discovering a model of the given data in form of an interpretable mathematical expression which  is made of constants, variables, and basic functions (e.g. addition, subtraction, multiplication, division, sine, exponential, etc.)

In the supervised learning setting, given a set of observations $ \left\{ ( x_i, y_i ) \right\} _{i=1}^N $ of the dependent and independent variables that characterize a phenomenon, the goal is to find a function $\hat{f}$ that approximates the true unknown function $ f:\mathbb{R}^n \rightarrow \mathbb{R} $ which governs the phenomenon $ y = f(x) $. The learning task is accomplished by minimizing a loss, e.g the mean squared error between the true and the predicted values.

Besides accuracy, to enforce interpretability it is preferable to have simple expressions that describe $ \hat{f} $. The most simple measure of complexity can be defined as the number of symbols, i.e. constants, variables, and basic functions, that appear in an expression.

### Multi Objective Optimization
To summarize, the problem, in a multi-objective optimization framework, can be formalized as follows
$$
    \hat{f} = \arg \min_{e \in \mathcal{E}} \left\{ L(e), C(e) \right\},
$$

where $\mathcal{E}$ is the space of mathematical expressions, $L$ is the loss function and $C$ is the complexity measure.

Given a set of candidate expressions $ \mathcal{S} $, to compare two candidate expressions $e, e' \in \mathcal{S}$,  we define the Pareto dominance relation as $e \prec e' $ which holds if and only if  
- $L(e) \le L(e')$;
- $C(e) \le C(e')$;
- at least one inequality is strict.

An expression is said to be Pareto-optimal if it is not dominated by any other expression. The set of all such non-dominated expressions defines the Pareto front:
$$
    \mathcal{P}_\mathcal{S}
    = \Big\{\, e \in \mathcal{S} \ \Big|\
    \nexists\, e' \in \mathcal{S} \text{ such that } e' \prec e \,\Big\}.
$$
In other words, $\mathcal{P}_\mathcal{S}$ contains all expressions for which no other expression is both more accurate and simpler.

In conclusion, we have to define a method to choose the best expression $e^*$ among the ones in $\mathcal{P}_\mathcal{S}$, this decisions is highly dependant on the specific problem that is considered.

### Genetic programming

The key aspect of genetic programming is that a problem can be solved by stochastically evolving a population of computer programs. It is possible to use the Genetic Programming approach to solve the Multi-Optimization Problem defined for Symbolic Regression.

The most fundamental type of computer program is a mathematical expression. A natural and effective way to represent a mathematical expression is using expression trees. An expression tree is, in the data structure sense, a tree characterized by the following properties:
- leaves represent constants and variables which appear in an expression
- internal nodes are the unary or binary functions.

In particular, nodes with one child node represent unary operators, nodes with two children are binary operators.

This structure has some desirable properties:
- the complexity of an expression is just the number of nodes in the tree
- the structure of an expression is easily adjustable.

Easily adjustable means that, given a syntactically correct expression, it is simple to create a new syntactically correct expression which is different from the first one. This property is particularly useful in the context of genetic operations. For example, if we want to make an expression simpler we can remove a sub-tree from its expression tree and substitute it with a constant. If we want to make an expression more complex we append a sub tree in place of a leaf node. Finally, if we want to make some internal changes in the tree, we have just to satisfy the arity constraints imposed by the number of children of a node.


## Introduction to PySR


### Problem

One field in which symbolic regression is a crucial task is physics.

Tycho Brahe (1546 - 1601) was a Danish astronomer. During his studies, he carefully collected a huge amount of measurements about the motion of the planets in the solar system. However, he could not find a law which could explain the patterns in the data that he was observing.

After Brahe's death, his assistant, Johannes Kepler (1571-1630), used the data collected by his master to find the three relations which are now called Three Laws of Planetary Motion.
In particular, for what regards the third law, Kepler investigated the relation between the semi-major axis of the orbit of each planet and the period of such orbit. For each planet, Kepler had available two measurements:
- the lenght of the semi-major axis $a$ (expressed in astronomical units AU);
- the period $T$ (in days)
of the orbit of each planet around the sun.

Kepler discoevered the third law in 1618: he performed symbolic regression ante-litteram. Now, starting from the Brahe's data, we try to re-discover the third law using PySR.

### Useful imports

In [1]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor

[juliapkg] Found dependencies: /Users/lucamanzoni/Documents/InProgress/Teaching/Global and Multi-Objective Optimization 2025/Notebooks/.pixi/envs/default/lib/python3.13/site-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /Users/lucamanzoni/Documents/InProgress/Teaching/Global and Multi-Objective Optimization 2025/Notebooks/.pixi/envs/default/lib/python3.13/site-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /Users/lucamanzoni/Documents/InProgress/Teaching/Global and Multi-Objective Optimization 2025/Notebooks/.pixi/envs/default/lib/python3.13/site-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia ^1.10.3
[juliapkg] Using Julia 1.12.2 at /opt/homebrew/bin/julia
[juliapkg] Using Julia project at /Users/lucamanzoni/Documents/InProgress/Teaching/Global and Multi-Objective Optimization 2025/Notebooks/.pixi/envs/default/julia_env
[juliapkg] Writing Project.toml:
           | [deps]
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
         

  Installing known registries into `~/.julia`
       Added `General` registry to ~/.julia/registries
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed IrrationalConstants ───────── v0.2.6
   Installed Adapt ─────────────────────── v4.4.0
   Installed Scratch ───────────────────── v1.3.0
   Installed MicroMamba ────────────────── v0.1.14
   Installed ScientificTypesBase ───────── v3.0.0
   Installed Tricks ────────────────────── v0.1.13
   Installed DiffRules ─────────────────── v1.15.1
   Installed DynamicExpressions ────────── v1.10.3
   Installed JSON3 ─────────────────────── v1.14.3
   Installed PtrArrays ─────────────────── v1.3.0
   Installed Preferences ───────────────── v1.5.0
   Installed MLJModelInterface ─────────── v1.11.1
   Installed TableTraits ───────────────── v1.0.1
   Installed DiffResults ───────────────── v1.1.0
   Installed PythonCall ────────────────── v0.9.26
   Installed ADTypes ───────────────────── v1.20.

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


### Data

In [2]:
planetary_data = pd.DataFrame(
    {
        "Planets":  ["Mercury", "Venus",   "Mars",     "Jupiter",  "Saturn",   "Uranus",   "Neptune"    ],
        "a":        [0.38710,   0.72333,    1.52366,    5.20336,    9.53707,    19.1913,    30.0690     ],
        "T":        [87.9693,   224.7008,   686.9796,   4332.8201,  10775.599,  30687.153,  60190.03    ]
    }
)

In the data set we do not consider the data of our planet Earth, we will use such data as a test set.

A possible assumption is that the semi-major axis length is the response variable and that the period is the independent variable.

In [3]:
X = planetary_data["T"].to_numpy().reshape(-1,1) # Dimensions (number of samples, number of features). In our case (7,1)
y = planetary_data["a"].to_numpy().reshape(-1,1) # Dimensions (number of samples, ). In our case (7,)

### Train

We are searching for a function $f$ such that
$$
    a ≈ f(T)
$$

Let's define the Symbolic Regression model using PySR.

In [4]:
model = PySRRegressor(
    maxsize=8, # Maximum number of nodes in the expression trees
    maxdepth=4, # Maximum depth of the trees
    niterations=200,
    populations=8, #PySR is a multi-population algorithm!

    # We do not know which operators are included in the true f. For this reason, we
    # consider a set of operators which introduces properties that are
    # rich enough in order to describe a physical phenomenon.

    binary_operators=[
        "+", "*",   # Linearity
        "^",        # Non-linearities: quadratic, cubic, root, division etc.
    ],
    unary_operators=[ # More non-linearities
        "exp", # Exponential growth/decay
        "sin", # Periodic behaviour
    ],

    # Each node in the expression tree has weight 1 in the complexity of the
    # complete expression. It is possible to set a different weight for the
    # operators as follows

    #complexity_of_operators = {"sin": 2, "exp": 3}

    # The search space is huge, so it is wise to introduce reasonable constraints
    # on the use of operators. For instance,
    #   exp(exp(sin(x^(exp(x)))))
    # is not something we would even want to compute.

    constraints={'^': (-1, 1)}, # The binary operator ^(x,y), i.e. x^y, is
                                # allowed to have argument x with unbounded
                                # complexity, but argument y with complexity 1.
                                # So y must be a single constant or a variable.
    nested_constraints={
        "exp": {"exp": 0, "sin": 0}, # nesting unary operators is not allowed
        "sin": {"exp": 0, "sin": 0},
    },

    # PySR uses MSE as loss function: L2DisLoss() . It is possible to change
    # the loss function used during fit by setting the following argument

    #elementwise_loss = L1DistLoss()   # L1 = |x-y|

    # Enabling reproducibility
    random_state=1618,
    deterministic=True,
    parallelism='serial'    # We have multipe populations, so it is possible
                            # to evolve them in parallel
)

In [5]:
sr_model = model.fit(
    X, # period T
    y, # semimajor axis length a
    variable_names = ["T"] # names of the independent variables
    )

/Users/lucamanzoni/Documents/InProgress/Teaching/Global and Multi-Objective Optimization 2025/Notebooks/.pixi/envs/default/lib/python3.13/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.085e+02  0.000e+00  y = 9.5193
3           5.227e+00  1.516e+00  y = T * 0.00053731
5           2.659e-04  4.943e+00  y = (T ^ 0.6652) * 0.019885
7           1.547e-04  2.709e-01  y = ((T ^ 0.66512) + -0.81639) * 0.019918
───────────────────────────────────────────────────────────────────────────────────────────────────
  - outputs/20251128_091017_HkAsE4/hall_of_fame.csv


### How is the best model chosen?

As we can see in the output of the fit procedure, at each iteration, the algorithm returns the Pareto front of candidate solutions. Now, we need a strategy to select the best model.

PySR uses as a default model selection strategy the score value. The score $s_i$ of expression $e_i$ is defined as
$$
s_i = - \frac{log L(e_i) - logL(e_{i-1})}{C(e_i) - C(e_{i-1})}
$$
where $L, C$ are the loss and complexity measures, respectively. In other words, $s_i$ tell us of how many orders of magnitude we decrease the loss by introducing a unit of complexity in the formulation of the expression. In conclusion, the expression with largest score $s_i$ is selected.

P.S. By default, PySR selects the candidate model with the highest score among expressions with a loss better than at least 1.5x the most accurate model.

### Interpreting the model


Symbolic Regression fits a model that is completely interpretable. Let's invesigate what is the functional form of the predicted model.

In [6]:
print(sr_model)

PySRRegressor.equations_ = [
	   pick     score                                      equation        loss  \
	0        0.000000                                      9.519328  108.451675   
	1        1.516270                              T * 0.0005373108    5.226617   
	2        4.943051                 (T ^ 0.66519815) * 0.01988481    0.000266   
	3  >>>>  0.270919  ((T ^ 0.6651178) + -0.8163918) * 0.019918196    0.000155   
	
	   complexity  
	0           1  
	1           3  
	2           5  
	3           7  
]


The equation discovered by PySR (the one with highest score) is
$$
    a = 0.02 \times T^{0.67}.
$$

Let's perform some algebraic manipulation, raise both sides to power $3$
$$
    a^3 = 0.02^3 \times T ^ {0.67 \times 3},
$$
and we get
$$
    a^3 = 2.0^{-6} \times T ^ {2}.
$$

In the end, we get the same relation discovered (by hand) by Kepler in 1618
$$
    a^3 \propto T^2 .
$$

To be exact, the constant $2.0^{-6}$ discovered by our model it is not very accurate. Using Newton's law of gravitation (published in 1687), the correct constant is
$$
    \frac{GM}{4π} ≈ 7.496 × 10^{-6},
$$
where $G$ is the gravitational constant and $M$ is the mass of the sun. Since our data set includes only $7$ observations, we cannot expect the model to be more accurate than this.

### Test

We can test the model by predicting the length of semi-major axis of Earth knowing that the period of its orbit is 365 days.


In [7]:
T_earth = np.array([[365]])
a_earth_pred = model.predict(T_earth) # predict uses the best expression found during fit
print("Predicted lenght of semi major axis of planet Earth: ", a_earth_pred[0], " AU.")

Predicted lenght of semi major axis of planet Earth:  0.9917762299017093  AU.


The true lenght of the semi major axis of Earth is $1.0$ AU. The trained model is accurate!

